# Object Detection, Hands-On: Teaching YOLO to See Something It Has Never Seen

We built a CNN from scratch and trained it on MNIST, which is how you come to
understand what a convolution does, but not how most people ship a vision feature.
Real projects start from a model somebody else already trained on millions of images,
and the first question is never "what architecture should I design", it is "does this
existing model already solve my problem, and if not, what is the cheapest way to make
it?"

This notebook walks that question end to end, on one deliberately awkward object: a
playing card.

1. **It works.** We run a pretrained YOLO detector on an ordinary scene and watch it
   find laptops and cups with no effort at all.
2. **How it works**, briefly, because the next part only makes sense if you know
   where the class names live inside the model.
3. **It fails.** We show it a playing card. It does not say "I don't know", it says
   `book`, confidently, because `book` is the closest word it owns.
4. **We fix it** the way you would actually fix it in a project: collect a few
   hundred labelled photographs, fine-tune a small YOLO on them, and measure whether
   the result is good enough.
5. **We measure it properly**, with precision, recall and mAP on a held-out test
   split, and then on a real photograph, which is where the interesting gap shows up.
6. **We test it live**, on a card held up to the webcam.


## 0. Setup

On Colab, the next cell installs the three libraries we need, Ultralytics for YOLO,
Gradio for the webcam apps and `datasets` for the training data, which takes about a
minute. Locally it detects that you are not on Colab and installs nothing, so activate
your course environment first.

A GPU makes the training in Section 5 roughly ten times faster. On Colab, switch it
on under **Runtime > Change runtime type > T4 GPU** before you go any further.

In [ ]:
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q ultralytics gradio datasets

print("running on Colab:", IN_COLAB)

In [ ]:
import base64
import io
import time
from pathlib import Path

import gradio as gr
import torch
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO

if torch.cuda.is_available():
    DEVICE = "cuda"  # Colab GPU runtime
elif torch.backends.mps.is_available():
    DEVICE = "mps"  # Apple Silicon
else:
    DEVICE = "cpu"

print("torch:", torch.__version__, "| device:", DEVICE)

### The three photographs

This notebook uses three photographs, and they are **embedded in the notebook itself**
as base64 text rather than shipped as separate files. That is why the next cell is a
wall of characters: it is the images, encoded. Run it and move on.

The reason for doing it this way is distribution. A single `.ipynb` runs anywhere you
can open it, including Colab, with no folder to copy alongside it and no download that
can fail, rate-limit or disappear. The cost is a larger file, which for three
photographs is a good trade.

All three come from Wikimedia Commons under free licences:

| Photograph | Author | Licence |
|---|---|---|
| [Coffee-desk-laptop-notebook](https://commons.wikimedia.org/wiki/File:Coffee-desk-laptop-notebook_%2824244320481%29.jpg) | www.Pixel.la Free Stock Photos | CC0 |
| [Ace playing cards](https://commons.wikimedia.org/wiki/File:Ace_playing_cards.jpg) | Enoch Lau | CC BY-SA 3.0 |
| [Ace-Ass](https://commons.wikimedia.org/wiki/File:Ace-Ass.JPG) | Edroeh | CC BY-SA 3.0 |

In [ ]:
# @title
# The three photographs, base64-encoded so this notebook is a single self-contained
# file. Nothing to read here; load_image() below turns these back into images.
IMAGE_B64 = {
    "desk": "...",
    "cards_in_hand": "...",
    "cards_on_table": "...",
}

In [ ]:
def load_image(name):
    """Decode one of the embedded photographs into a PIL image."""
    return Image.open(io.BytesIO(base64.b64decode(IMAGE_B64[name]))).convert("RGB")


print("embedded photographs:", ", ".join(IMAGE_B64))

## 1. A detector that already works

A **classifier** takes an image and returns one label for the whole picture. A
**detector** returns a list, and for each object it found: a class name, a confidence
score, and a bounding box `(x0, y0, x1, y1)` in pixel coordinates. That list can be
empty, and it can contain the same class more than once.

YOLO ("You Only Look Once") is the detector family you will meet most often outside
research, because it is fast enough for video and small enough for a laptop. We use
`yolo11n.pt`, the "nano" size: 5.4 MB of weights, downloaded automatically the first
time you construct it. Those weights were trained on **COCO**, a dataset of 80
everyday object classes, and those 80 names are the entire vocabulary this model can
ever produce.

In [ ]:
detector = YOLO("yolo11n.pt")

class_names = detector.names  # dict: class index -> class name
print("classes this model can output:", len(class_names))
print()
print(sorted(class_names.values()))

In [ ]:
def detect(image, model=None, conf=0.25):
    """Run a YOLO model and return detections as a plain list of dicts."""
    model = model or detector
    result = model.predict(image, conf=conf, device=DEVICE, verbose=False)[0]
    detections = [
        {
            "label": model.names[int(box.cls)],
            "conf": float(box.conf),
            "box": tuple(round(v, 1) for v in box.xyxy[0].tolist()),
        }
        for box in result.boxes
    ]
    return sorted(detections, key=lambda d: -d["conf"])


def show_detections(image, detections, color="red"):
    """Draw boxes and labels on a copy of the image."""
    annotated = image.copy()
    draw = ImageDraw.Draw(annotated)
    font = ImageFont.load_default(size=max(14, image.size[1] // 32))
    for det in detections:
        x0, y0, x1, y1 = det["box"]
        draw.rectangle([x0, y0, x1, y1], outline=color, width=4)
        caption = f"{det['label']} {det['conf']:.2f}"
        draw.rectangle(
            [x0, y0 - font.size - 6, x0 + len(caption) * font.size * 0.6, y0],
            fill=color,
        )
        draw.text((x0 + 3, y0 - font.size - 4), caption, fill="white", font=font)
    return annotated


def print_detections(detections, title):
    print(f"{title}: {len(detections)} detections")
    for det in detections:
        print(f"  {det['label']:<14} conf {det['conf']:.3f}  box {det['box']}")

In [ ]:
desk_image = load_image("desk")

desk_detections = detect(desk_image)
print_detections(desk_detections, "desk photo")

show_detections(desk_image, desk_detections)

`laptop` at 0.939, `vase` at 0.762, `cup` at 0.735 and two overlapping `potted plant`
boxes. This is what a detector working well looks like, and it took no configuration
at all.

It is also worth noticing what is already slightly wrong, because we will need the
vocabulary for it later. The plant pot gets labelled both `vase` (0.762) and `cup`
(0.414) on nearly the same box; COCO contains both classes, they genuinely overlap in
the real world, and the model hedges. The plant itself gets two boxes, at 0.518 and
0.510, rather than one. Neither is a bug. A detector produces a *list of guesses with
scores*, and turning that list into a decision, which threshold, which duplicates to
suppress, is your job, not the model's.

### Your object, live

Now do it with something in the room. The cell below starts a small
[Gradio](https://www.gradio.app/) app: a few lines of Python that spin up a web app
rendered inside the notebook output, with `gr.Image(sources=["webcam", "upload"])`
giving you both a camera and a file picker. Capture a frame and press **Submit**.

**What to hold up.** COCO knows an object in the *poses it was photographed in*, so
some choices work far better than others. Two reliable ones:

- **A smartphone held up to the camera.** `cell phone` is a COCO class and this is
  the single most dependable live demo: around 0.90 confidence, plus a free `person`
  detection for your hand and arm.
- **A mug or cup standing on the desk**, in view of the camera rather than in your
  hand. On a desk, `cup` scores around 0.95. Held in a fist it often scores nothing
  at all, because your hand hides most of the shape and COCO's cup photographs are
  mostly of cups sitting on surfaces.

Less reliable, and interesting for exactly that reason: a water bottle (`bottle`)
usually works standing up and often fails in a hand; a book held open (`book`)
frequently returns nothing; scissors get called `knife` about as often as
`scissors`.

In [ ]:
CAPTURE_PATH = Path("captured.jpg")


def capture_and_detect(image):
    """Save the frame so later cells can reuse it, and run the COCO detector on it."""
    if image is None:
        return (
            None,
            "No image received. Capture a frame or upload a file, then press Submit.",
        )

    image = image.convert("RGB")
    image.save(CAPTURE_PATH, quality=95)
    detections = detect(image)
    if not detections:
        report = "**Nothing detected.** No COCO class scored above 0.25 anywhere in this frame."
    else:
        report = "\n".join(f"- `{d['label']}` {d['conf']:.2f}" for d in detections)
    return show_detections(image, detections), report


coco_demo = gr.Interface(
    fn=capture_and_detect,
    inputs=gr.Image(
        sources=["webcam", "upload"], type="pil", label="Hold something up"
    ),
    outputs=[gr.Image(label="Detections"), gr.Markdown(label="What YOLO saw")],
    title="Pretrained YOLO11, 80 COCO classes",
    flagging_mode="never",
)

# share=True also prints a public https://....gradio.live URL. In VS Code that link is
# clickable and opens a real browser tab, which is much the easiest way to get webcam
# permission granted; on Colab it sidesteps the notebook iframe. The link is random,
# lasts 72 hours and tunnels through Gradio's servers, so close the app when you finish.
coco_demo.launch(share=True)

### 1.1 Your turn

1. Run `detect` on `desk_image` with `conf=0.01` and print the results. How many
   detections do you get, and how many of the extra ones are right?
2. Write `count_by_class(detections)` that returns a `dict` of class name to count,
   and call it on both the `conf=0.25` and `conf=0.01` results.
3. Pick any three class names from `class_names.values()` that you would expect to
   find in this room, and say for each whether you would trust it at 0.25.

The point of (1) is to settle a reflex before Section 3: when a detector misses
something, the first instinct is always to lower the threshold, and it is worth
seeing what that actually costs.

In [ ]:
# TODO: 1) detect(desk_image, conf=0.01) and print it
# TODO: 2) write count_by_class(detections) and apply it to both thresholds
raise NotImplementedError

## 2. How YOLO works, in one page

Three ideas from this are needed to follow the rest of the notebook.

**One pass, not a search.** Older detectors worked by proposing thousands of candidate
regions and classifying each one, which meant running a classifier thousands of times
per image. YOLO's insight, the one in its name, is to make the whole thing a single
forward pass: the image goes through a convolutional backbone once, and the output
layer directly predicts boxes for every position at once. That is why it runs on
video on a laptop, and it is the reason detection became practical for ordinary
products rather than research demos.

**Dense predictions, then suppression.** The network's output is a grid over the
image, at several scales so that both small and large objects have a grid cell sized
for them. Every cell predicts box coordinates plus a score for each class. That
produces far more boxes than there are objects, most with near-zero scores, and
several genuine ones clustered on the same object. **Non-maximum suppression** (NMS)
cleans this up afterwards: sort boxes by confidence, keep the best one, discard
anything overlapping it by more than an IoU threshold, repeat. The duplicate
`potted plant` boxes in Section 1 are two survivors of that process: they overlap
each other just little enough that neither suppressed the other.

**The class list is part of the architecture.** This is the idea that matters for
Section 3. Those per-cell class scores are literally output channels of the final
convolution: 80 classes means the head produces 80 class numbers per position, fixed
when the model was built and filled with meaning when it was trained. There is no
lookup table of words to extend and no prompt to edit. Adding `playing_card` to this
model means constructing a new head with a different output shape and training it,
which is exactly what Section 5 does.

## 3. Now the playing card

Here is a real photograph of a hand holding four aces up to a camera, the same thing
you are about to do in class. Before you run the cell, predict the output: does a
model with 80 everyday object classes return nothing, or something wrong?

In [ ]:
card_photo = load_image("cards_in_hand")

card_detections = detect(card_photo)
print_detections(card_detections, "photo of four aces")

show_detections(card_photo, card_detections)

`person` at 0.715 and `book` at 0.618.

The `person` box is correct and a good sign: the hand and forearm really are there,
and the model has no trouble with them. The interesting output is `book`, a single
box drawn around the whole fan of cards with 0.618 confidence, which in most
production thresholds counts as a solid detection.

This is the failure mode to remember, and it is worse than silence. The model is not
blind to the cards, it located them perfectly well. It has no *word* for them, so it
reached for the nearest one it owns: a flat, pale, rectangular, papery object held in
a hand is, among 80 available choices, most like a book. A logging pipeline that
records only the top label would tell you a student was holding a book, with
confidence, and nothing anywhere in the output would hint that the correct answer was
not on the menu.

Run the app from Section 1 again with an actual card, or upload a photo, and you will
get your own version of this: usually `book`, sometimes `cell phone`, occasionally
`remote`, occasionally nothing.

### 3.1 Your turn

1. Search `class_names.values()` for any name containing `"card"`, and print the
   result.
2. Run `detect` on `card_photo` at `conf=0.01` and print everything. Does anything
   card-shaped appear if you look hard enough down the list?
3. In one sentence: is this a threshold problem, a model-size problem, or something
   else, and what would you have to change to fix it?

Keep your answer to (3) in mind for Section 5, because it is the whole argument for
what we do next.

In [ ]:
# TODO: 1) print class names containing "card"
# TODO: 2) detect(card_photo, conf=0.01) and print everything
raise NotImplementedError

Nothing in the 80 names is remotely card-like, and dropping the threshold to 0.01
adds only more wrong labels on the same pixels. It is not a threshold problem: no
score can appear for an output channel that does not exist. It is not a model-size
problem either; the largest YOLO11 ever released still has exactly these 80 classes,
because that is what COCO contains. The only fix is to change the model's
vocabulary, which means training.

## 4. A ready-made dataset

To make a detector that outputs `playing_card` we need labelled examples: photographs
with a box drawn around every card in them. Collecting and labelling those is where
most of the real effort in an applied vision project goes, and it is also the part
most likely to be already done for you. Before you organise a labelling session, spend
twenty minutes looking, because public datasets exist for a surprising number of
problems.

This one comes from **Roboflow 100**, a benchmark suite of a hundred small real-world
detection datasets, and we pull it from the Hugging Face Hub as
`Francesco/poker-cards-cxcvz`. It is about 60 MB: 964 training, 128 validation and
193 test photographs of playing cards, four or five per image, shot on sofas,
bedspreads, carpets and tables, at angles, under ordinary indoor lighting. Which is
to say: exactly the kind of photograph you would have spent an afternoon collecting.

### What YOLO expects on disk

Ultralytics does not read Hugging Face datasets, so we convert. The target format is
deliberately boring: images in one folder tree, labels in a parallel one with the same
filenames, and a `data.yaml` naming the splits and the classes.

```
poker_cards/
    data.yaml
    images/train/00000.jpg     labels/train/00000.txt
    images/val/00000.jpg       labels/val/00000.txt
    images/test/00000.jpg      labels/test/00000.txt
```

Each label file holds one line per object, and an image with no objects gets an empty
file:

```
<class_index> <centre_x> <centre_y> <width> <height>
```

All four numbers are **fractions of the image size**, not pixels, so
`0 0.5 0.5 0.2 0.3` means "class 0, centred in the middle, one fifth of the image
wide and three tenths tall". Normalising this way means a label survives resizing the
image, which matters because training resizes everything to a fixed square anyway.

The source dataset stores boxes the other common way, COCO style: absolute pixel
`[x, y, width, height]` measured from the top-left corner. Converting between the two
is the three lines of arithmetic in the middle of the next cell, and getting that
arithmetic subtly wrong is such a common bug that we will draw the results back onto
the images afterwards to check.

In [ ]:
from datasets import load_dataset

DATASET = Path("poker_cards")
HF_SPLITS = {"train": "train", "val": "validation", "test": "test"}


def build_dataset():
    """Convert the Hugging Face dataset into the layout Ultralytics expects."""
    hf = load_dataset("Francesco/poker-cards-cxcvz")

    for split, hf_split in HF_SPLITS.items():
        (DATASET / "images" / split).mkdir(parents=True, exist_ok=True)
        (DATASET / "labels" / split).mkdir(parents=True, exist_ok=True)
        cards = 0
        for row in hf[hf_split]:
            stem = f"{row['image_id']:05d}"
            row["image"].convert("RGB").save(
                DATASET / "images" / split / f"{stem}.jpg", quality=92
            )

            width, height = row["width"], row["height"]
            lines = []
            for x, y, w, h in row["objects"]["bbox"]:  # COCO: pixels, top-left corner
                lines.append(  # YOLO: fractions, centre point
                    f"0 {(x + w / 2) / width:.6f} {(y + h / 2) / height:.6f} "
                    f"{w / width:.6f} {h / height:.6f}"
                )
            (DATASET / "labels" / split / f"{stem}.txt").write_text(
                "\n".join(lines) + "\n"
            )
            cards += len(lines)
        print(f"{split:<6} {len(hf[hf_split]):>4} images, {cards:>5} cards")

    data_yaml = DATASET / "data.yaml"
    data_yaml.write_text(
        f"path: {DATASET.resolve()}\n"
        "train: images/train\n"
        "val: images/val\n"
        "test: images/test\n"
        "names:\n"
        "  0: playing_card\n"
    )
    return data_yaml


DATA_YAML = build_dataset()
print()
print(DATA_YAML.read_text())

### Starting simple: one class

The source dataset labels every card with which card it is, all 52 of them, and the
conversion above collapses that into a single class called `playing_card`. Every box
becomes class `0`, and the model we train will answer one question: **is there a card
here, and where?**

That is deliberate. "Is there a card" and "which card is it" are very different
problems, and the first is the one worth solving first: it is learnable from this much
data, its metrics are readable, and it is the part a product usually needs anyway,
because you can crop the card and read it with something else afterwards. Starting
with the narrow question and widening it later is almost always cheaper than starting
wide.

Section 8 returns to this with the obvious next step, a model that also names the
**suit**, which is four classes rather than one or fifty-two.

### Look at the data

Never trust a dataset you have not looked at, including this conversion. The cell
below reads each label file back, converts the fractions to pixels, and draws them on
the image. This is the check that catches the whole family of swapped-coordinate,
forgot-to-normalise and off-by-one bugs the format invites.

In [ ]:
def load_labels(split, stem):
    """Read a YOLO label file back into pixel boxes."""
    image = Image.open(DATASET / "images" / split / f"{stem}.jpg")
    width, height = image.size
    boxes = []
    for line in (
        (DATASET / "labels" / split / f"{stem}.txt").read_text().strip().split("\n")
    ):
        if not line:
            continue
        _class_index, cx, cy, w, h = (float(v) for v in line.split())
        boxes.append(
            (
                (cx - w / 2) * width,
                (cy - h / 2) * height,
                (cx + w / 2) * width,
                (cy + h / 2) * height,
            )
        )
    return image.convert("RGB"), boxes


def label_sheet(split, count=6, columns=3, thumb=320):
    """A contact sheet of images with their labels drawn on."""
    stems = sorted(p.stem for p in (DATASET / "images" / split).glob("*.jpg"))[:count]
    rows = (len(stems) + columns - 1) // columns
    sheet = Image.new("RGB", (columns * thumb, rows * thumb), "black")
    for position, stem in enumerate(stems):
        image, boxes = load_labels(split, stem)
        draw = ImageDraw.Draw(image)
        for box in boxes:
            draw.rectangle(list(box), outline="lime", width=4)
        sheet.paste(
            image.resize((thumb, thumb)),
            ((position % columns) * thumb, (position // columns) * thumb),
        )
    return sheet


label_sheet("train")

Tight boxes around every card, including the heavily overlapping ones in a fanned
hand, on real backgrounds with real shadows. Note how much harder this looks than the
neat studio photographs you might have imagined: cards are tilted, partly hidden
behind each other, sometimes blurred, sometimes half off the edge of the frame. That
difficulty is the point. A model trained on easy pictures is a model that works on
easy pictures.

### 4.1 Your turn

Get a feel for the dataset before you train on it, which is a habit worth having.

1. Count the cards per image across the training split and print the distribution
   (how many images have 4 cards, how many have 5, and so on).
2. Print the mean box area as a fraction of the image area. Objects that occupy a
   large fraction of the frame are much easier to detect than small ones, and this
   number is the single best predictor of how hard a detection dataset will be.
3. Call `label_sheet("test")` and compare it to the training sheet. Do the test
   photographs look like they came from the same place as the training ones?

Question (3) is the one that matters in Section 6. If the answer is "yes, obviously
the same", a good test score tells you less than you would like.

In [ ]:
# TODO: 1) distribution of cards per image across the training split
# TODO: 2) mean box area as a fraction of image area
# TODO: 3) label_sheet("test") and compare it with the training sheet
raise NotImplementedError

## 5. Training the model

We are not training from scratch. `YOLO("yolo11n.pt")` loads the COCO weights from
Section 1, and training on top of them is **fine-tuning**: the backbone already knows
how to turn pixels into edges, corners, textures and "pale rectangular thing on a
darker surface", because it learned that from 118,000 photographs of other objects.
What has to change is mostly the detection head, whose output shape goes from 80
classes to 1, plus some adjustment in the later layers of the backbone.

That distinction is why 964 images is a plausible number. Training a
2.6-million-parameter detector from random initialisation on 964 images would produce
something barely usable; starting from COCO, it is a few minutes of work. Exercise 5.1
makes you check that claim rather than take it on trust.

The arguments worth knowing:

- `epochs=15`, how many times we sweep the whole training set. Too few and it has not
  learned; too many and it starts memorising these particular images.
- `imgsz=640`, the square size every image is resized to. Larger sees smaller objects
  and costs more time and memory. These images are already 640x640.
- `batch=16`, images per parameter update. Drop it to 8 or 4 if you run out of memory.
- `seed=0`, so repeated runs are roughly comparable.
- `deterministic=False`, which needs a word of explanation. Ultralytics defaults this
  to `True`, which asks PyTorch to refuse any operation without a reproducible
  implementation. Several operations in the backward pass have no such implementation
  on Apple GPUs, so the default prints a warning for *every batch* and buries the
  training output. Turning it off silences that, and costs nothing we had: as the note
  below says, this training was never going to be bit-for-bit reproducible anyway.
  `seed=0` still fixes the shuffling and augmentation, which is the part that matters.

Ultralytics also augments while training, flips, colour jitter and mosaic (stitching
four training images into one), which is why the `train_batch*.jpg` files in the
output folder look so strange. That augmentation is free extra variation and it is
doing real work.

**This takes about ten minutes** on an Apple Silicon GPU or a Colab T4, and several
times that on CPU. If you are on CPU, or just impatient, set `epochs=5`: this dataset
is easy enough that one epoch already gets most of the way, and the later epochs are
mostly buying tighter boxes rather than more cards found.

In [ ]:
card_model = YOLO("yolo11n.pt")

train_metrics = card_model.train(
    data=str(DATA_YAML),
    epochs=15,
    imgsz=640,
    batch=16,
    device=DEVICE,
    project=str(
        Path.cwd() / "runs"
    ),  # otherwise this lands wherever Ultralytics' settings point
    name="cards",
    exist_ok=True,
    seed=0,
    deterministic=False,  # see the note below
)

RUN_DIR = Path(card_model.trainer.save_dir)
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
print()
print("run directory:", RUN_DIR)
print("best weights: ", BEST_WEIGHTS, f"({BEST_WEIGHTS.stat().st_size / 1e6:.1f} MB)")
print("classes:", YOLO(BEST_WEIGHTS).names)

One thing to notice before we go on: the model now has **exactly one class**. It is
not the COCO model plus cards, it is a new model that has forgotten all 80 original
classes, because we replaced the head. If a product
needs both "is there a card" and "is there a phone", you either train one model on a
dataset labelled for both, or you run two models. That surprises people, and it is
worth deciding which you want before you collect data.

One practical note on the numbers from here on. GPU training is not bit-for-bit
reproducible even with a fixed seed, so figures quoted in the text will be close to
yours rather than identical. Nothing in the argument depends on the third decimal
place; if something differs in the first, that is worth investigating.

In [ ]:
Image.open(RUN_DIR / "results.png")

The top row is the three losses measured on the training data, the bottom row repeats
them on the validation split and adds the metrics. The three losses are worth knowing
by name, because a detector is really three predictions stacked together and each loss
scores one of them:

- **`box_loss`** scores *where*. For every prediction matched to a real object, it
  measures how badly the predicted rectangle overlaps the labelled one, using an
  IoU-based distance: identical boxes cost nothing, boxes that miss cost a lot. This
  is the loss that makes boxes tight, and it is the one that drives the mAP50-95
  number in Section 6.
- **`cls_loss`** scores *what*. For each prediction it asks how confidently the right
  class was chosen, penalising confident wrong answers far more than hesitant ones.
  With a single class this collapses to something close to "is there an object here at
  all", which is why it starts small here and falls fast. On a 52-class problem it
  would be doing most of the work.
- **`dfl_loss`**, distribution focal loss, is a refinement of `box_loss`. Rather than
  regressing each box edge to a single number, YOLO predicts a small probability
  distribution over possible positions for each edge and takes its expectation, which
  turns out to be easier to optimise and more accurate at sub-pixel precision.
  `dfl_loss` scores that distribution. You will rarely act on it directly; watch it
  only to confirm it is falling with the others.

What you want to see is what you do see: all three falling and flattening, and
crucially the **validation** curves tracking the training curves downward rather than
turning back upward. A validation loss that bottoms out and then climbs while the
training loss keeps falling is the signature of overfitting, and it is the signal to
stop training or get more data.

Notice how fast it gets good. Validation mAP50 is already about 0.989 after the
**first** epoch. Fine-tuning a pretrained detector onto a new class of large,
high-contrast objects is close to the easiest thing you can ask of this technology.

### 5.1 Does the pretraining actually matter?

That claim about fine-tuning deserves testing rather than assuming, and it costs one
more training run. `YOLO("yolo11n.yaml")` builds the same network from its
*architecture description* instead of its weights: identical layers, identical size,
random initialisation, no COCO knowledge whatsoever. Same data, same epochs, same
everything else.

This is another ten minutes. Start it and read ahead; we will use both models in
Section 6.

In [ ]:
scratch_model = YOLO("yolo11n.yaml")  # architecture only: random weights, no COCO

scratch_metrics = scratch_model.train(
    data=str(DATA_YAML),
    epochs=15,
    imgsz=640,
    batch=16,
    device=DEVICE,
    project=str(Path.cwd() / "runs"),
    name="cards_scratch",
    exist_ok=True,
    seed=0,
    deterministic=False,  # see the note below
)

print()
print(f"{'run':<24}{'mAP50':>9}{'mAP50-95':>11}")
for label, metrics in [
    ("fine-tuned from COCO", train_metrics),
    ("from scratch", scratch_metrics),
]:
    results = metrics.results_dict
    print(
        f"{label:<24}{results['metrics/mAP50(B)']:>9.3f}{results['metrics/mAP50-95(B)']:>11.3f}"
    )

Same architecture, same data, same budget, and a large gap. The pretrained backbone is
not a minor optimisation, it is most of why this works at a thousand images. Everything
the from-scratch model needs to learn about edges, corners and "pale rectangle on a
darker background", the fine-tuned one already knew from 118,000 photographs of buses
and sofas and dogs.

Worth thinking about before Section 6: how many labelled photographs do you think
from-scratch training would need to close that gap, and what does that tell you about
when "just train your own model" is and is not good advice?

## 6. The metrics: what they measure and why you should care

This is the part of the notebook worth remembering, because the models change every
few months and the measurement does not.

Ultralytics printed validation numbers after every epoch and we picked the best
checkpoint using them, so those numbers are no longer an honest estimate of anything:
we used them to make a choice. That is what the **test split** is for. It was set
aside before training and nothing in the process has touched it.

Start with why there is no single "accuracy" here. For a classifier, accuracy is
well defined: the model gives one answer per image, and it is right or wrong.
A detector gives a *list* per image, and that list can be wrong in four different
ways. It can miss an object. It can report an object that is not there. It can find
the object but call it the wrong class. It can find the object, call it correctly, and
draw the box in the wrong place. One number cannot summarise four failure modes, so
detection uses several, and each one is blind to something the others catch.

### 6.1 IoU

Before you can count anything, you need a rule for when a predicted box counts as
matching a labelled one. They are never pixel-identical, so "equal" is useless.

**Intersection over Union** is the rule: the area where the two boxes overlap, divided
by the total area they cover between them.

```
IoU = area(overlap) / area(union)
```

It is 1.0 for identical boxes, 0.0 for boxes that do not touch, and roughly 0.5 for
two boxes you would describe as "the same object, noticeably different extents".
Fixing a threshold, conventionally 0.5, turns a fuzzy judgement into a yes/no decision
we can count.

**Why you should care:** the IoU threshold silently sets how much sloppiness you are
willing to accept in *where* the object is. If you are counting cards on a table, a
loose box is fine and 0.5 is generous enough. If a robot arm is going to reach for the
thing in the box, a box that is 50% off is useless, and you should be measuring at 0.8
or higher. Nobody can pick that number for you, because it depends on what happens
next with the box.

### 6.2 Precision and recall

With IoU as a matching rule, every prediction and every label lands in one of three
buckets. Matching is done greedily in confidence order, and each labelled box can only
be claimed once, which is what makes a second box on an already-detected card count
against you rather than for you.

- **True positive (TP)**: a prediction that matched an unclaimed label at IoU above
  the threshold.
- **False positive (FP)**: a prediction that matched nothing. Either an object that is
  not there, or a duplicate box on an object already found.
- **False negative (FN)**: a labelled object that no prediction claimed. A miss.

From those:

```
precision = TP / (TP + FP)      of everything the model reported, how much was real
recall    = TP / (TP + FN)      of everything that was there, how much did it find
```

**Why you should care, and why you need both.** They fail in opposite directions and
each is trivially easy to game alone. A model that reports nothing at all has no false
positives, so its precision is undefined or perfect, and its recall is zero. A model
that carpets the image in boxes will find everything, so its recall is 1.0, and its
precision is near zero. Quoting one without the other is the oldest trick in the
vendor-benchmark book.

Which one you optimise is a **business decision, not a modelling decision**, and it
comes from the relative cost of the two errors:

- **Recall matters most when a miss is expensive.** Detecting a pedestrian in front of
  a vehicle, a tumour on a scan, a weapon at a checkpoint. A false alarm is an
  annoyance; a miss is a catastrophe. Buy recall and pay for it in precision.
- **Precision matters most when a false alarm is expensive.** Automatically issuing a
  parking fine, flagging an account for fraud review, auto-rejecting a part on a
  production line. Every false positive costs someone real money or real time, so you
  would rather quietly miss a few.

Recall has one more property worth internalising: **it is the metric that catches a
model failing silently.** A model that stops detecting anything produces no wrong
answers, so precision looks fine and an error-rate dashboard shows nothing at all. If
you are only going to monitor one number in production, monitor recall.

**F1** is the harmonic mean of the two, for when you are forced to produce a single
number. It is fine as a summary and dangerous as a target, because it implicitly
declares a false positive and a false negative equally costly, which is true for
almost no real application.

In [ ]:
test_metrics = card_model.val(
    data=str(DATA_YAML),
    split="test",
    device=DEVICE,
    project=str(Path.cwd() / "runs"),
    name="cards_test",
    exist_ok=True,
)

print()
for key, value in test_metrics.results_dict.items():
    print(f"  {key:<26} {value:.4f}")

**Precision 0.994, recall 0.995, mAP50 0.995, mAP50-95 0.910** on 838 cards across 193
unseen images. Read those in the language of 6.2: of everything it reported, 99.4% was
a real card; of every card that was there, it found 99.5%. The mAP50-95 of 0.910,
noticeably below the mAP50 of 0.995, says the boxes are very slightly loose, which for
counting cards does not matter at all and for a robot picking them up might.

These are excellent numbers and they are about to become the most interesting thing in
the notebook, because they are also somewhat misleading.

### 6.3 The confidence threshold is a dial you own

Every detection carries a confidence score, and everything above is reported while
everything below is discarded. That cutoff is *yours*, not the model's, and moving it
slides you along the precision-recall trade-off without retraining anything.

The sweep below re-evaluates the same weights on the same test images at several
cutoffs. Nothing about the model changes between rows.

In [ ]:
print(f"{'conf':>6}{'precision':>12}{'recall':>9}{'F1':>8}")
for threshold in [0.25, 0.50, 0.75, 0.90, 0.95]:
    metrics = card_model.val(
        data=str(DATA_YAML),
        split="test",
        conf=threshold,
        device=DEVICE,
        project=str(Path.cwd() / "runs"),
        name="sweep",
        exist_ok=True,
        plots=False,
        verbose=False,
    )
    precision = metrics.results_dict["metrics/precision(B)"]
    recall = metrics.results_dict["metrics/recall(B)"]
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    print(f"{threshold:>6.2f}{precision:>12.3f}{recall:>9.3f}{f1:>8.3f}")

Something surprising happens here, and it is worth pausing on because it is not what
the textbook trade-off looks like.

Between 0.25 and 0.50 **nothing moves at all**: precision 0.994 and recall 0.995 at
both. At 0.75 recall has barely dipped, to 0.989. Then between 0.90 and 0.95 the floor
falls away: recall goes 0.888, then 0.075. At 0.99 the model reports nothing and every
number is zero.

The reason is that this model's confidences are bunched very tightly. On its own test
data the median detection scores 0.927 and four out of five score above 0.9, so almost
every cutoff below 0.9 keeps almost everything and every cutoff above it keeps almost
nothing. The precision-recall trade-off is real, but on easy, in-distribution data it
is compressed into a narrow band you can easily miss.

Two things follow. First, **this table is what you put in front of a client**, not a
single number, because picking the row is their decision and depends on what each kind
of mistake costs them; reading the F1 column and taking the maximum is the lazy answer
and usually the wrong one. Second, and less obviously, a flat table like this is a
warning sign rather than a reassurance. A model this confident everywhere has told you
almost nothing about which of its answers to doubt, and Section 6.5 shows what that
costs the moment the images stop looking like the test set.

### 6.4 mAP: the number on every model's spec sheet

Reporting precision and recall at one hand-picked cutoff makes models hard to compare,
because whoever picks the cutoff picks the winner. **Average Precision** removes the
choice: sweep the confidence threshold across every value the model produced, plot
precision against recall at each step, and take the area under that curve. One number
per class, no threshold to argue about. **mAP** is the mean of AP across classes, so
with our single class, AP and mAP are the same number.

The two you will see quoted:

- **mAP50** uses an IoU threshold of 0.5 throughout. It asks "did you find the
  objects", and it is forgiving about exactly where the box sits.
- **mAP50-95** averages AP over ten IoU thresholds from 0.50 to 0.95 in steps of 0.05.
  It asks "did you find the objects *and* is the box tight". It is always the lower
  number, and the gap between the two tells you how sloppy your boxes are.

**Why you should care:** mAP is the right metric for *comparing* models and the wrong
metric for *operating* one. It deliberately averages over the threshold you will
actually have to choose, so a model can win on mAP and still be the wrong choice at
the cutoff your application needs. Use mAP to pick a model, then use the Section 6.3
table to configure it.

And watch the gap. Ours is 0.995 against 0.910: the model essentially always finds the
card, and its boxes are good but not surgical. A large mAP50 with a much smaller
mAP50-95 means a model that knows roughly where things are and is careless about
extents, which matters enormously if something downstream crops, measures or grabs the
object, and not at all if you are just counting them.

In [ ]:
Image.open(Path(test_metrics.save_dir) / "BoxPR_curve.png")

That curve is what mAP50 is the area of. It stays pinned in the top-right corner,
high precision maintained at high recall, until it drops away at the very end, which
is the handful of cards no threshold ever finds.

### 6.5 Does it work outside the test set?

Our test images were taken by the same people, with the same cameras, of the same
decks, on the same sofas, in the same session as the training images. Holding them back
proves the model did not memorise individual photographs. It does not prove the model
works on *your* card, in *your* room, under *your* lighting.

So let's check, on the two photographs from Section 3. The model has never seen them
and they come from a different source, a different deck and a different decade to its
training data. Both contain exactly four cards.

In [ ]:
real_photos = {
    "hand holding four aces": load_image("cards_in_hand"),
    "four aces on a wooden table": load_image("cards_on_table"),
}

models = {
    "COCO pretrained": detector,  # 80 classes, never saw a card label
    "trained from scratch": scratch_model,
    "fine-tuned": card_model,
}

for name, photo in real_photos.items():
    print(name)
    for label, model in models.items():
        detections = detect(photo, model=model, conf=0.25)
        found = (
            ", ".join(f"{d['label']} {d['conf']:.2f}" for d in detections) or "nothing"
        )
        print(f"  {label:<22} {len(detections)} detections: {found}")
    print()

In [ ]:
table_photo = real_photos["four aces on a wooden table"]
show_detections(table_photo, detect(table_photo, model=card_model, conf=0.25))

Three models, the same two photographs, and three different kinds of answer.

The **COCO pretrained** model does what Section 3 showed: it has no word for a card, so
it reaches for `book` and `person`. The **from-scratch** model, trained on exactly the
same card photographs as the fine-tuned one, finds little or nothing, and what it does
report it reports weakly. The **fine-tuned** model finds **eight cards out of eight**,
one box each, nothing spurious.

That three-way split is the whole argument of the notebook in one output. The
pretrained model fails because of its vocabulary. The from-scratch model fails despite
having exactly the right vocabulary and exactly the right data, because a thousand
images is nowhere near enough to learn vision from nothing. Only the combination, other
people's features plus your labels, actually works.

And the fine-tuned model transfers, which deserves stating plainly: a thousand real
photographs with real lighting, angles and occlusion produced a detector that works on
photographs from a completely different source. That is the payoff for training on
genuinely varied data.

Now look at the scores rather than the count, because this is where it gets
interesting. On its own test data this model's detections have a median confidence of
0.927 and 80% of them clear 0.9. On these two photographs the median is 0.870, the
weakest correct detection scores **0.59**, and only two of the eight clear 0.9.

Accuracy survived the move to new data. **Calibration did not.** The model is still
right, and it is markedly less sure, and it has no way of telling you which of those
two things has changed.

That matters because of the dial in Section 6.3. Suppose you had read that table and
chosen a cutoff of 0.90, which looked almost free: recall 0.888, precision a perfect
1.000. Apply that same cutoff to these two real photographs and you keep **2 of the 8
cards**, a recall of 0.250. Same model, same threshold, same task, and the number you
would have quoted to a client overstates real-world recall by a factor of three and a
half.

Nothing was implemented wrong. The metric answered exactly the question it was asked,
which was "how well does this reproduce these labels on images like these", and we
read it as "how well does this find playing cards". That substitution is a *scoping*
error rather than a modelling one, and it is the most common way a vision project
passes its own metrics and then disappoints in the room.

The practical habit is cheap: whenever you report a vision metric, say in the same
breath what data it was measured on, and if that data did not come from where the
system will actually be used, say that too. Then go and collect a few dozen images that
*did* come from there and measure again. It is a morning's work, and it is the
difference between a number and a claim.

### 6.6 Your turn

Build the comparison the section just described, so you have done it once yourself.

1. Run the detector on both real photographs at confidence 0.90, 0.75, 0.50 and 0.25,
   and display each result with `show_detections`.
2. At each threshold, count by eye how many detections actually sit on a card. That is
   your true positives; anything else is a false positive, and 8 minus the true
   positives is your misses, since there are 4 cards in each of the 2 photographs.
3. Tabulate recall on these two photographs next to the test-split recall at the same
   threshold from 6.3, and look at the two columns side by side.
4. Pick the threshold you would actually ship for "count the cards on the table", and
   say in one sentence what you are buying and what you are paying for.

Counting by eye is not a workaround here, it is the exercise: it is what "collect a
small test set from your own distribution" feels like at the smallest possible scale,
and it takes about four minutes. Two photographs is far too small a sample to conclude
anything, which is itself worth feeling.

In [ ]:
# TODO: 1) detect() on both real photos at conf 0.5 / 0.25 / 0.1 / 0.05, show each
# TODO: 2) count by eye how many detections land on a card
# TODO: 3) tabulate precision and recall over both photos (8 cards in total)
raise NotImplementedError

## 7. The live demo: both models, one webcam

Time to hold up a card. The app below runs **both** detectors on the same frame: the
original COCO model, which will reach for `book` or `cell phone`, and the model we
just trained, which has one word and actually means it.

`share=True` makes Gradio print a public `https://....gradio.live` URL as well as the
local one. In VS Code that link is clickable and opens in a real browser tab, which is
the easiest way to get webcam permission to work, and on Colab it sidesteps the
notebook iframe entirely. The link is random, lives for 72 hours, and tunnels through
Gradio's relay servers, so treat anything you point the camera at as leaving your
machine, and call `card_demo.close()` when you are done.

This is also the honest test of everything above: your webcam, your room and your deck
are all things this model has never seen.

In [ ]:
def compare_models(image):
    """Run the COCO detector and the fine-tuned card detector on the same frame."""
    if image is None:
        return (
            None,
            None,
            "No image received. Capture a frame or upload a file, then press Submit.",
        )

    image = image.convert("RGB")
    image.save(CAPTURE_PATH, quality=95)

    coco_hits = detect(image, model=detector, conf=0.25)
    card_hits = detect(image, model=card_model, conf=0.25)

    lines = ["### Pretrained YOLO11 (80 COCO classes)"]
    if coco_hits:
        lines += [f"- `{d['label']}` {d['conf']:.2f}" for d in coco_hits]
    else:
        lines.append("- *nothing above 0.25*")

    lines.append("### Our fine-tuned model (1 class)")
    if card_hits:
        lines.append(
            f"- **{len(card_hits)} card(s)**: "
            + ", ".join(f"{d['conf']:.2f}" for d in card_hits)
        )
    else:
        lines.append("- *no cards above 0.25*")

    return (
        show_detections(image, coco_hits, color="orange"),
        show_detections(image, card_hits, color="red"),
        "\n".join(lines),
    )


card_demo = gr.Interface(
    fn=compare_models,
    inputs=gr.Image(
        sources=["webcam", "upload", "clipboard"], type="pil", label="Show me a card"
    ),
    outputs=[
        gr.Image(label="COCO model"),
        gr.Image(label="Our card model"),
        gr.Markdown(label="Both models"),
    ],
    title="Before and after fine-tuning",
    description="The same frame through the pretrained detector and through the one we just trained.",
    flagging_mode="never",
)

card_demo.launch(share=True)

The upload and clipboard inputs matter as much as the webcam here: go and find
photographs of cards online, paste them in, and you have an instant test set from a
distribution neither of us chose. That is a much harsher test than your own desk, and
it is free.

Things worth trying, because each one is a gap you can see with your own eyes:

- A card **flat on the desk**, photographed steeply from the side, so it is a
  trapezoid rather than a rectangle.
- A card **half hidden** behind your hand or another card.
- A **fan of cards** held like a poker hand. The training data is full of these, so it
  should do well, which is the point.
- **No card at all**: your phone, a business card, a folded receipt, an empty desk.
  Anything reported here is a false positive.
- The **card back** instead of the face, and a card from a deck with a very different
  design, a tarot deck or a game like Uno.
- A **photograph of a screen** showing cards, or a card **drawn on paper**.

### 7.1 The same thing on live video

Pressing Submit for every frame is not what a detector is for. The app below runs on
the webcam **continuously**, so you can wave a card around and watch the box follow it.

Two changes make that happen. `streaming=True` on the input tells Gradio to keep
sending frames from the camera instead of waiting for a button, and `live=True` on the
Interface tells it to call the function on each one. Two more arguments are worth
setting explicitly, because the defaults will surprise you:

- `stream_every=0.1` asks for a frame every tenth of a second. The default is 0.5,
  which looks like a slideshow.
- `time_limit=600` is how long one stream may run before Gradio stops it. The default
  is 30 seconds, which will cut you off mid-demonstration.

This is also where the "one forward pass" claim from Section 2 stops being an
abstraction. The status line reports how long inference plus drawing actually took for
each frame, and on a laptop GPU it is usually **under 10 ms**, which is a theoretical
ceiling somewhere around a hundred frames a second. You will not see anything like
that rate, and the gap is the point: the bottleneck is the round trip from browser to
server and back, not the model. A detector this size is genuinely not the slow part of
a video pipeline.

In [ ]:
def stream_detect(frame):
    """Run the card detector on one webcam frame. Gradio calls this ~10x a second."""
    if frame is None:
        return None, "waiting for camera"

    started = time.perf_counter()
    result = card_model.predict(frame, conf=0.25, device=DEVICE, verbose=False)[0]
    elapsed_ms = (time.perf_counter() - started) * 1000

    # Ultralytics draws its own annotations and hands back a BGR array; flip to RGB.
    annotated = result.plot()[:, :, ::-1].copy()
    count = len(result.boxes)
    label = "no cards" if count == 0 else f"{count} card" + ("" if count == 1 else "s")
    return annotated, f"{label}  |  {elapsed_ms:.0f} ms per frame"


live_demo = gr.Interface(
    fn=stream_detect,
    inputs=gr.Image(sources=["webcam"], streaming=True, type="numpy", label="Camera"),
    outputs=[gr.Image(label="Live detections"), gr.Textbox(label="Status")],
    title="Live card detection",
    description="The fine-tuned detector, running continuously on the webcam.",
    live=True,
    stream_every=0.1,  # the default of 0.5 looks like a slideshow
    time_limit=600,  # the default of 30 seconds cuts you off mid-demo
    flagging_mode="never",
)

live_demo.launch(share=True)

Worth doing while it runs, because motion shows you things a still frame hides:

- Move a card slowly across the frame and watch the box track it, then move it fast and
  watch the box lag or drop out entirely. Motion blur is a kind of image this model has
  barely seen.
- Rotate a card through 360 degrees in the plane of the camera. There is usually an
  angle where confidence dips.
- Walk the card towards and away from the camera. Somewhere far away it stops being
  detected, and that distance is a real specification if anyone ever asks "how far can
  it see?".
- Hold it still and watch the confidence number jitter frame to frame even though
  nothing is moving. The model has no memory between frames; each one is a fresh,
  independent guess. Production video systems smooth this with tracking, which is what
  Ultralytics' `model.track()` adds on top of exactly this loop.

If the stream is choppy, it is almost certainly the `share=True` tunnel rather than the
model. Open the local `http://127.0.0.1:...` URL instead and it will be noticeably
smoother.

### 7.2 Your turn

Turn the detector into something that makes a decision, because a product cannot ship
a list of confidence scores.

1. Write `count_cards(image, conf=0.25, min_area=0.005)` that runs the card model and
   returns the number of detections, ignoring any box smaller than `min_area` as a
   fraction of the image area. Tiny boxes are nearly always noise, and filtering by
   size is the cheapest false-positive filter there is.
2. Have it return a verdict string too: `"no cards detected"`, `"1 card"` or
   `"N cards"`.
3. Plug it into a copy of `compare_models`, relaunch, and hold up one card, then two,
   then none. Does the count ever disagree with what you can see?

Then: given Section 6.3, what confidence threshold did you choose for `count_cards`,
and what would you have to change if the cost of overcounting were much higher than
the cost of undercounting?

In [ ]:
# TODO: write count_cards(image, conf=0.25, min_area=0.005) -> (count, verdict string)
# TODO: plug it into a copy of compare_models and relaunch the app
raise NotImplementedError

When you are finished, shut the apps down. Each `launch(share=True)` holds open a
tunnel, and the streaming one keeps a camera stream alive as well, so a notebook that
has been re-run a few times can leave several running at once.

In [ ]:
for app in (coco_demo, card_demo, live_demo):
    app.close()
print("apps closed")

## 8. Where this leaves us

| | Pretrained YOLO11 (COCO) | Our fine-tuned YOLO11 |
|---|---|---|
| Classes | 80 everyday objects | 1: `playing_card` |
| On a photo of cards | `book`, 0.62 | all 8 of 8 found, 0.59-0.93 |
| Test split | mAP50 0.95 on COCO | mAP50 0.995, mAP50-95 0.910 |
| Labelled images needed | none | 964, already labelled by somebody else |
| Time to get it | one line | ten minutes of training |
| What it would have cost | nothing | an afternoon, had the dataset not existed |

Four things to carry out of this.

**A model's class list is a product constraint, not a detail.** The first question
about any pretrained vision model is what set of answers it can physically produce,
compared to the set your use case needs. Sections 1 to 3 were decided entirely by that
question, and it is the one most often skipped at the start of a project.

**Look for the dataset before you build one.** We got 1,285 labelled photographs for
the cost of a download. Public detection datasets exist for retail shelves, road
signs, PPE compliance, crop disease, wildlife, medical imaging and a long tail of
oddities. Searching for one is twenty minutes; collecting and labelling your own is
weeks.

**But look *at* the data too.** This one's boxes were excellent and its 52 class names
were substantially wrong, and nothing downstream would have told us. Every metric in
Section 6 measures agreement with the labels, so label noise is invisible to all of
them and shows up only when you crop a few boxes and read them.

**Your metric is only as broad as your test set.** Precision, recall and mAP are
precise answers to exactly the question "how well does this model reproduce these
labels on these images". Turning that into "how well does this work" is a claim about
where the images came from, and it is yours to justify. Our own version of this was
sharp: a confidence cutoff of 0.90 costs almost nothing on the test split, recall
0.888, and costs three quarters of the cards on two ordinary photographs, recall 0.250.

**Distribution shift shows up in confidence before it shows up in accuracy.** The
model found every card in both outside photographs and was visibly less sure while
doing it, median confidence dropping from 0.927 to 0.870 and the weakest correct
detection scoring 0.59. Accuracy held; calibration slipped. Which is why a threshold
tuned on one distribution is not transferable to another, and why monitoring the
*distribution* of confidence scores in production is an unusually cheap early warning.

And one option we deliberately did not use: a **cloud Vision API**. You POST an image
and get labels, text and objects back from a vocabulary far larger than COCO's, with
no training and no weights to host. It is a real capability jump and a trade, not an
upgrade: you pay per call, you add a network round trip, you cannot inspect or fix the
model, it can change under you without notice, and you are sending your images to
somebody else's servers, which for photographs of people or documents pulls GDPR and
data residency into scope before the prototype rather than after. Which option is
right, a pretrained model, a fine-tuned one, or an API, is not answerable from
anybody's spec sheet. It is answerable with the measurements you just ran, on your own
images, with your own ground truth.

## 9. If there is time: a model that names the suit

Everything so far answered one question, "is there a card". The obvious next question
is "which card", and the first step toward it is the **suit**: four classes instead of
one, from exactly the same photographs and exactly the same boxes.

This changes the shape of the problem rather than its size. Detecting a card means
finding a pale rectangle, which is mostly about edges. Naming its suit means reading a
small symbol printed on that rectangle, which is a genuinely harder visual task, and it
lets the model be wrong in a new way: right box, wrong label. Section 6's metrics are
about to earn their keep, because with four classes mAP is an average over them, and an
average can hide one class doing badly.

Two practical notes. The source dataset has one class name that came through corrupted,
a bare `59` on 111 objects, with no recoverable suit; the cleanest fix is to drop the
images containing those, which costs about 9% of the data and leaves no unlabelled
cards behind. And "Trefoils" is this dataset's word for clubs.

### 9.1 Your turn

1. Build a second YOLO dataset in a new folder, `poker_cards_suits`, from the same
   Hugging Face source. Keep the same boxes, but take each label's class from the
   **last word** of the card's name, so `A Spades` and `7 Spades` both become
   `Spades`. Skip any image containing an object whose category id is 0 or 1.
2. Write a `data.yaml` listing the four suit names in a fixed order.
3. Fine-tune `yolo11n.pt` on it for 15 epochs, exactly as in Section 5.
4. Evaluate on the test split and print **AP50 per class**, not just the mean, using
   `metrics.box.ap50` alongside `metrics.box.ap_class_index`.

Then compare. Is mAP50 higher or lower than the one-class model's, and would you have
predicted that before running it? Which suit comes out hardest? And what would this
same exercise look like with all 52 classes?

In [ ]:
# TODO: 1) build poker_cards_suits/ with 4 classes from the last word of each card name,
#          skipping any image that contains a category id of 0 or 1
# TODO: 2) write its data.yaml with the four suit names
# TODO: 3) fine-tune yolo11n.pt on it for 15 epochs
# TODO: 4) evaluate on the test split and print AP50 per class
raise NotImplementedError

Read the per-class column, not just the mean. That is the habit this section exists to
build: **mAP is an average, and an average hides its worst member.** A four-class model
reporting a healthy overall mAP50 can be carrying one class that barely works, and
printing the classes separately is the only way to see it. On a 52-class version the
effect would be far stronger, because the rarest classes would have a handful of
examples each and their AP would be close to noise while the mean stayed respectable.

There is a second thing worth knowing, and it is part of why the one-class model came
first. Put a few of this dataset's annotated crops next to the names it gives them and
you will find the card names are not entirely reliable, suits included. Boxes that are
right and names that are sometimes wrong put a ceiling on any model trained from those
names, and no metric in Section 6 can see it, because every one of them measures
agreement with the labels rather than with reality. Which is the note to end on: **a
metric is a measurement of your labels, and your labels are a claim about the world
that somebody has to check.**